<a href="https://colab.research.google.com/github/jkj05/BIS5/blob/main/PSO_ACO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random

n = 30
iterations = 300
w = 0.7
c1 = 1.5
c2 = 1.5

demand = 150

low = [20, 20]
high = [80, 80]

cost = [
    (500, 5.3, 0.004),
    (400, 5.5, 0.006),
    (200, 5.8, 0.009)
]

def allocate(x):
    p1 = max(20, min(80, x[0]))

    lower = max(20, demand - p1 - 80)
    upper = min(80, demand - p1 - 20)

    p2 = max(lower, min(upper, x[1]))
    p3 = demand - p1 - p2

    return [p1, p2, p3]

def fitness(x):
    return sum(
        a + b * p + c * p * p
        for (a, b, c), p in zip(cost, allocate(x))
    )

pos = [[random.uniform(20, 80) for _ in range(2)] for _ in range(n)]
vel = [[random.uniform(-2, 2) for _ in range(2)] for _ in range(n)]

pbest = [p[:] for p in pos]
pfit = [fitness(p) for p in pos]

best = pbest[pfit.index(min(pfit))][:]
best_fit = fitness(best)

for _ in range(iterations):
    for i in range(n):
        for j in range(2):
            vel[i][j] = (
                w * vel[i][j]
                + c1 * random.random() * (pbest[i][j] - pos[i][j])
                + c2 * random.random() * (best[j] - pos[i][j])
            )

            pos[i][j] += vel[i][j]
            pos[i][j] = max(low[j], min(high[j], pos[i][j]))

        f = fitness(pos[i])

        if f < pfit[i]:
            pbest[i] = pos[i][:]
            pfit[i] = f

        if f < best_fit:
            best = pos[i][:]
            best_fit = f

power = allocate(best)

print("\nOptimal Power Allocation")

for i, p in enumerate(power):
    print(f"Generator {i + 1}: {p:.2f} MW")

print("--------------------------------")
print(f"Total Generation: {sum(power):.2f} MW")
print(f"Minimum Fuel Cost: {best_fit:.2f}")


Optimal Power Allocation
Generator 1: 80.00 MW
Generator 2: 50.00 MW
Generator 3: 20.00 MW
--------------------------------
Total Generation: 150.00 MW
Minimum Fuel Cost: 1959.20


In [ ]:
import random
import math

random.seed(42)

cities = [(0, 0), (2, 6), (5, 3), (8, 7), (6, 1)]
names = ['A', 'B', 'C', 'D', 'E']


n = len(cities)
ants = 20
iterations = 100
alpha = 1
beta = 5
rho = 0.5
q = 100

dist = [[0] * n for _ in range(n)]

for i in range(n):
    for j in range(n):
        x1, y1 = cities[i]
        x2, y2 = cities[j]
        dist[i][j] = math.hypot(x2 - x1, y2 - y1)

pher = [[1] * n for _ in range(n)]

def choose_city(cur, unvisited):
    weights = [
        pher[cur][j] ** alpha * (1 / dist[cur][j]) ** beta
        for j in unvisited
    ]

    pick = random.random() * sum(weights)
    total = 0

    for j, weight in zip(unvisited, weights):
        total += weight
        if total >= pick:
            return j

    return unvisited[-1]

def route_length(route):
    return sum(
        dist[route[i]][route[(i + 1) % n]]
        for i in range(n)
    )

best_route = None
best_dist = float('inf')

for _ in range(iterations):
    routes = []

    for _ in range(ants):
        route = [0]
        unvisited = list(range(1, n))

        while unvisited:
            nxt = choose_city(route[-1], unvisited)
            route.append(nxt)
            unvisited.remove(nxt)

        length = route_length(route)
        routes.append((route, length))

        if length < best_dist:
            best_dist = length
            best_route = route[:]

    for i in range(n):
        for j in range(n):
            pher[i][j] *= (1 - rho)

    for route, length in routes:
        deposit = q / length

        for i in range(n):
            a = route[i]
            b = route[(i + 1) % n]
            pher[a][b] += deposit
            pher[b][a] += deposit

print("Best Delivery Route:")
print(" -> ".join(names[i] for i in best_route + [best_route[0]]))
print("Minimum Distance:", round(best_dist, 3))

Best Delivery Route:
A -> E -> C -> D -> B -> A
Minimum Distance: 25.726
